In [1]:
#install packages, you can also use pip install -e .
%pip install yfinance
%pip install pandas
%pip install swig
%pip install numpy
%pip install wrds
%pip install pyportfolioopt
%pip install datetime
%pip install requests

#Install finrl library
%pip install git+https://github.com/AI4Finance-Foundation/FinRL.git

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
  Cloning https://github.com/AI4Finance-Foundation/FinRL.git to /tmp/pip-req-build-_chng3xo
  Running command git clone --filter=blob:none --quiet https://github.com/AI4Finance-Foundation/FinRL.git /tmp/pip-req-build-_chng3xo
  Resolved https://github.com/AI4Finance-Foundation/FinRL.git to commit 2334a5fe6d30629157f13c3b0319e1637e15e123
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... don

In [2]:
#Import packages

import pandas as pd
import numpy as np 
import datetime
import yfinance as yf 
import requests

from finrl.meta.preprocessor.yahoodownloader import YahooDownloader
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl import config_tickers
from finrl.config import INDICATORS

import itertools

In [3]:
#Generate List of S&P-500 Tickers

# Source - https://stackoverflow.com/a/75845569
# Posted by Asclepius, modified by community. See post 'Timeline' for change history
# Retrieved 2026-09-07, License - CC BY-SA 4.0

def list_slickcharts_sp500() -> pd.DataFrame:
    # Ref: https://stackoverflow.com/a/75845569/
    url = 'https://www.slickcharts.com/sp500'
    user_agent = 'Mozilla/5.0 (X11; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/111.0'  # Default user-agent fails.
    response = requests.get(url, headers={'User-Agent': user_agent})
    return pd.read_html(response.text, match='Symbol', index_col='Symbol')[0]

df = list_slickcharts_sp500() #Schreibt die erstellte Tabelle in einen Datenframe
df.head() #Zeigt die Struktur der Tabelle
print(df.columns.tolist()) #Gibt die Spaltennamen an -> Debugging
tickers = df.index.str.replace(".", "-", regex=False).tolist() # Schreibt die Indizes in eine Liste, sodass nur noch die Ticker da sind
print(tickers) #Gibt die Liste zur Überprüfung aus -> Debugging

['#', 'Company', 'Weight', 'Price', 'Chg', '% Chg']
['NVDA', 'AAPL', 'MSFT', 'AMZN', 'GOOGL', 'GOOG', 'AVGO', 'META', 'TSLA', 'MU', 'BRK-B', 'LLY', 'JPM', 'WMT', 'AMD', 'V', 'JNJ', 'XOM', 'MA', 'INTC', 'ORCL', 'ABBV', 'BAC', 'CSCO', 'PLTR', 'CVX', 'COST', 'LRCX', 'KO', 'CAT', 'MRK', 'AMAT', 'UNH', 'GE', 'MS', 'PG', 'DELL', 'NFLX', 'HD', 'GS', 'PM', 'WFC', 'PANW', 'RTX', 'SNDK', 'GEV', 'ANET', 'KLAC', 'AMGN', 'TXN', 'C', 'TMO', 'IBM', 'AXP', 'LIN', 'CRWD', 'CRM', 'VZ', 'APH', 'MRVL', 'TMUS', 'STX', 'SCHW', 'PEP', 'ABT', 'GILD', 'DE', 'DIS', 'MCD', 'QCOM', 'T', 'ADI', 'NEE', 'BLK', 'UNP', 'WELL', 'WDC', 'BA', 'PFE', 'COP', 'ETN', 'UBER', 'NOW', 'DHR', 'TJX', 'BKNG', 'VRTX', 'BMY', 'NEM', 'COF', 'GLW', 'CB', 'SPGI', 'PLD', 'ISRG', 'PGR', 'CVS', 'PH', 'LMT', 'MDT', 'SBUX', 'SYK', 'MO', 'LOW', 'FTNT', 'ACN', 'BNY', 'ADP', 'HOOD', 'MPC', 'VRT', 'APP', 'ABNB', 'VLO', 'ADBE', 'CEG', 'MCK', 'FCX', 'HWM', 'EQIX', 'BX', 'PSX', 'SO', 'CME', 'USB', 'TT', 'PNC', 'GD', 'KKR', 'CMCSA', 'PWR', 'DUK', '

/tmp/ipykernel_100626/148648598.py:12: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  return pd.read_html(response.text, match='Symbol', index_col='Symbol')[0]


In [4]:
#Debugging
ticker_test = {"AAPL", "MSFT"}

In [5]:
#Setup für den Yahoo Downloader. Zeiträume werden später gesplitet
TRAIN_START_DATE = '2009-01-01'
TRAIN_END_DATE = '2020-09-01' 
TRADE_START_DATE = '2020-09-01' #Eher Test zur Validierung statt Trading
TRADE_END_DATE = '2026-09-07'

In [ ]:
#Daten herunterladen mit dem Yahoodownloader über den gesamten Zeitraum und den 503 Tickersymbolen
df_raw = YahooDownloader(start_date= TRAIN_START_DATE,
                         end_date = TRADE_END_DATE,
                         ticker_list = tickers).fetch_data()

YF deprecation warning: set proxy via new config function: yf.set_config(proxy=proxy)


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed

1 Failed download:
['JPM']: DNSError('Failed to perform, curl: (6) Could not resolve host: fc.yahoo.co

In [7]:
#Daten anssehen
df_raw.head()

Price,date,close,high,low,open,volume,tic,day
0,2009-01-02,10.259194,10.297098,9.798037,9.854891,4236220,A,4
1,2009-01-02,2.714299,2.722974,2.547105,2.568640,746015200,AAPL,4
2,2009-01-02,17.199024,17.292149,16.816896,17.189391,13163193,ABT,4
3,2009-01-02,7.287052,7.395878,7.226829,7.395878,3445200,ACGL,4
4,2009-01-02,24.225481,24.319016,23.412451,23.412451,3787400,ACN,4


In [8]:
#Daten mit technischen Indikatoren erweitern
fe = FeatureEngineer(use_technical_indicator=True, #Berechnung der technischen Indikatoren aktivieren
                     tech_indicator_list = INDICATORS, #Vorgefertigte Liste an Idikatoren übergeben
                     use_vix = True, #Volatilitätsindex aktivieren
                     use_turbulence=True, #Turbulance Index aktivieren
                     user_defined_feature = False) #Usereigene Features aus

semi_processed_data = fe.preprocess_data(df_raw)

Successfully added technical indicators


[*********************100%***********************]  1 of 1 completed


Shape of DataFrame:  (4421, 8)
Successfully added vix
Successfully added turbulence index


In [9]:
semi_processed_data

,date,close,high,low,open,volume,tic,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2009-01-02,10.259194,10.297098,9.798037,9.854891,4236220,A,4,0.000000,10.924279,9.941555,100.000000,66.666667,100.000000,10.259194,10.259194,39.189999,0.000000
1,2009-01-02,2.714299,2.722974,2.547105,2.568640,746015200,AAPL,4,0.000000,10.924279,9.941555,100.000000,66.666667,100.000000,2.714299,2.714299,39.189999,0.000000
2,2009-01-02,17.199024,17.292149,16.816896,17.189391,13163193,ABT,4,0.000000,10.924279,9.941555,100.000000,66.666667,100.000000,17.199024,17.199024,39.189999,0.000000
3,2009-01-02,7.287052,7.395878,7.226829,7.395878,3445200,ACGL,4,0.000000,10.924279,9.941555,100.000000,66.666667,100.000000,7.287052,7.287052,39.189999,0.000000
4,2009-01-02,24.225481,24.319016,23.412451,23.412451,3787400,ACN,4,0.000000,10.924279,9.941555,100.000000,66.666667,100.000000,24.225481,24.225481,39.189999,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1829875,2026-07-30,78.230003,80.830002,77.879997,80.830002,7171000,XEL,3,-0.032799,81.965028,77.873972,47.338658,-79.902999,5.874092,79.994334,79.346630,17.090000,1953.076751
1829876,2026-07-30,155.960144,156.764931,153.535839,154.092234,12578100,XOM,3,3.613730,161.032051,132.069075,58.362322,125.514862,27.945883,143.298454,146.328570,17.090000,1953.076751
1829877,2026-07-30,157.000000,166.160004,155.470001,162.000000,3308000,YUM,3,-1.340789,170.427703,140.296296,52.500878,51.448567,33.730248,155.196667,153.325533,17.090000,1953.076751
1829878,2026-07-30,94.959999,96.480003,93.489998,96.169998,2178000,ZBH,3,1.651183,96.574773,85.739228,55.868352,141.172178,21.775728,90.262766,87.471375,17.090000,1953.076751


In [11]:
#DataFrame nach Tag und Ticker sortieren

list_ticker = semi_processed_data["tic"].unique().tolist()
list_date = list(pd.date_range(semi_processed_data['date'].min(),semi_processed_data['date'].max()).astype(str))
combination = list(itertools.product(list_date,list_ticker))

fully_processed_data = pd.DataFrame(combination, columns=["date","tic"]).merge(semi_processed_data,on=["date", "tic"],how="left")
fully_processed_data = fully_processed_data[fully_processed_data['date'].isin(semi_processed_data['date'])]
fully_processed_data = fully_processed_data.sort_values(['date','tic'])

fully_processed_data = fully_processed_data.fillna(0)

fully_processed_data.head()

,date,tic,close,high,low,open,volume,day,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma,vix,turbulence
0,2009-01-02,A,10.259194,10.297098,9.798037,9.854891,4236220.0,4.0,0.0,10.924279,9.941555,100.0,66.666667,100.0,10.259194,10.259194,39.189999,0.0
1,2009-01-02,AAPL,2.714299,2.722974,2.547105,2.568640,746015200.0,4.0,0.0,10.924279,9.941555,100.0,66.666667,100.0,2.714299,2.714299,39.189999,0.0
2,2009-01-02,ABT,17.199024,17.292149,16.816896,17.189391,13163193.0,4.0,0.0,10.924279,9.941555,100.0,66.666667,100.0,17.199024,17.199024,39.189999,0.0
3,2009-01-02,ACGL,7.287052,7.395878,7.226829,7.395878,3445200.0,4.0,0.0,10.924279,9.941555,100.0,66.666667,100.0,7.287052,7.287052,39.189999,0.0
4,2009-01-02,ACN,24.225481,24.319016,23.412451,23.412451,3787400.0,4.0,0.0,10.924279,9.941555,100.0,66.666667,100.0,24.225481,24.225481,39.189999,0.0


In [13]:
#Speicherung der Daten

#Aufteilung in 2 seperate Listen
train = data_split(fully_processed_data, TRAIN_START_DATE, TRAIN_END_DATE)
trade = data_split(fully_processed_data, TRADE_START_DATE, TRADE_END_DATE)
print(len(train))
print(len(trade))



1215504
614376


In [14]:
train.to_csv("../data/train.csv", index=False)
trade.to_csv("../data/trade.csv", index=False)